In [ ]:
# PyTorch is normally pre-installed in Google Colab.
# Install PyTorch Geometric and Neo4j driver.

!pip -q install torch-geometric neo4j pandas matplotlib seaborn

In [ ]:
import os
import time
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid
from torch_geometric.nn import (
    GCNConv,
    SAGEConv,
    GATConv,
    GINConv
)

from neo4j import GraphDatabase

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

In [ ]:
dataset = Planetoid(
    root="/content/Cora",
    name="Cora"
)

data = dataset[0]

print(dataset)
print(data)
print()
print("Nodes:", data.num_nodes)
print("Edges:", data.num_edges)
print("Features:", data.num_features)
print("Classes:", dataset.num_classes)

In [ ]:
print("Node feature shape:", data.x.shape)
print("Edge index shape:", data.edge_index.shape)
print("Labels shape:", data.y.shape)

print("\nTraining nodes:", int(data.train_mask.sum()))
print("Validation nodes:", int(data.val_mask.sum()))
print("Test nodes:", int(data.test_mask.sum()))

In [ ]:
# Graph DB configuraiton

In [ ]:
from getpass import getpass

NEO4J_URI = input("Neo4j URI: ")
NEO4J_USERNAME = input("Neo4j username: ")
NEO4J_PASSWORD = getpass("Neo4j password: ")
NEO4J_DATABASE = input("Neo4j database [neo4j]: ") or "neo4j"

driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD)
)

driver.verify_connectivity()

print("Neo4j connection successful!")

In [ ]:
features = data.x.cpu().numpy()
labels = data.y.cpu().numpy()

edges = data.edge_index.cpu().numpy().T

print("Features:", features.shape)
print("Edges:", edges.shape)

In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:
    session.run("""
        MATCH (n:CoraPaper)
        DETACH DELETE n
    """)

print("Previous Cora data removed.")

In [ ]:
BATCH_SIZE = 250

node_records = []

for i in range(data.num_nodes):
    node_records.append({
        "id": int(i),
        "label": int(labels[i]),
        "features": features[i].tolist()
    })

for start in range(0, len(node_records), BATCH_SIZE):

    batch = node_records[start:start + BATCH_SIZE]

    with driver.session(database=NEO4J_DATABASE) as session:
        session.run("""
            UNWIND $rows AS row
            CREATE (:CoraPaper {
                id: row.id,
                label: row.label,
                features: row.features
            })
        """, rows=batch)

print("Cora nodes inserted:", len(node_records))

In [ ]:
edge_records = [
    {
        "source": int(src),
        "target": int(dst)
    }
    for src, dst in edges
]

for start in range(0, len(edge_records), BATCH_SIZE):

    batch = edge_records[start:start + BATCH_SIZE]

    with driver.session(database=NEO4J_DATABASE) as session:
        session.run("""
            UNWIND $rows AS row
            MATCH (a:CoraPaper {id: row.source})
            MATCH (b:CoraPaper {id: row.target})
            CREATE (a)-[:CITES]->(b)
        """, rows=batch)

print("Cora relationships inserted:", len(edge_records))

In [ ]:
with driver.session(database=NEO4J_DATABASE) as session:

    result = session.run("""
        MATCH (n:CoraPaper)
        RETURN count(n) AS nodes
    """)

    print("Neo4j nodes:", result.single()["nodes"])

    result = session.run("""
        MATCH ()-[r:CITES]->()
        RETURN count(r) AS relationships
    """)

    print("Neo4j relationships:", result.single()["relationships"])

In [ ]:
paper_id = 0

with driver.session(database=NEO4J_DATABASE) as session:

    result = session.run("""
        MATCH (p:CoraPaper {id: $paper_id})
              -[:CITES]->
              (neighbor:CoraPaper)

        RETURN neighbor.id AS id,
               neighbor.label AS label
        LIMIT 10
    """, paper_id=paper_id)

    for record in result:
        print(record.data())

In [ ]:
data = data.to(device)

print(data)

In [ ]:
from torch_geometric.nn import MessagePassing

class BasicGNNConv(MessagePassing):

    def __init__(self, in_channels, out_channels):
        super().__init__(aggr="mean")

        self.lin = nn.Linear(
            in_channels,
            out_channels
        )

    def forward(self, x, edge_index):

        return self.propagate(
            edge_index,
            x=x
        )

    def message(self, x_j):

        return x_j

    def update(self, aggr_out):

        return self.lin(aggr_out)


class BasicGNN(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim
    ):
        super().__init__()

        self.conv1 = BasicGNNConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = BasicGNNConv(
            hidden_dim,
            output_dim
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [ ]:
class GCN(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim
    ):
        super().__init__()

        self.conv1 = GCNConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = GCNConv(
            hidden_dim,
            output_dim
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [ ]:
class GraphSAGE(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim
    ):
        super().__init__()

        self.conv1 = SAGEConv(
            input_dim,
            hidden_dim
        )

        self.conv2 = SAGEConv(
            hidden_dim,
            output_dim
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [ ]:
class GAT(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim,
        heads=4
    ):
        super().__init__()

        self.conv1 = GATConv(
            input_dim,
            hidden_dim,
            heads=heads,
            dropout=0.5
        )

        self.conv2 = GATConv(
            hidden_dim * heads,
            output_dim,
            heads=1,
            concat=False,
            dropout=0.5
        )

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.elu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [ ]:
class GIN(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim,
        output_dim
    ):
        super().__init__()

        mlp1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim)
        )

        mlp2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, output_dim)
        )

        self.conv1 = GINConv(mlp1)
        self.conv2 = GINConv(mlp2)

    def forward(self, x, edge_index):

        x = self.conv1(x, edge_index)
        x = F.relu(x)

        x = F.dropout(
            x,
            p=0.5,
            training=self.training
        )

        x = self.conv2(x, edge_index)

        return x

In [ ]:
def train_model(
    model,
    data,
    epochs=200,
    lr=0.01,
    weight_decay=5e-4
):

    model = model.to(device)

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    criterion = nn.CrossEntropyLoss()

    start_time = time.perf_counter()

    best_val_acc = 0.0
    best_test_acc = 0.0

    for epoch in range(1, epochs + 1):

        model.train()

        optimizer.zero_grad()

        out = model(
            data.x,
            data.edge_index
        )

        loss = criterion(
            out[data.train_mask],
            data.y[data.train_mask]
        )

        loss.backward()

        optimizer.step()

        model.eval()

        with torch.no_grad():

            pred = out.argmax(dim=1)

            train_acc = (
                pred[data.train_mask]
                == data.y[data.train_mask]
            ).float().mean().item()

            val_acc = (
                pred[data.val_mask]
                == data.y[data.val_mask]
            ).float().mean().item()

            test_acc = (
                pred[data.test_mask]
                == data.y[data.test_mask]
            ).float().mean().item()

        if val_acc > best_val_acc:

            best_val_acc = val_acc
            best_test_acc = test_acc

    total_time = time.perf_counter() - start_time

    return {
        "best_val_accuracy": best_val_acc,
        "test_accuracy_at_best_val": best_test_acc,
        "training_time_seconds": total_time,
        "parameters": sum(
            p.numel()
            for p in model.parameters()
            if p.requires_grad
        )
    }

In [ ]:
models = {

    "Basic GNN": BasicGNN(
        data.num_features,
        64,
        dataset.num_classes
    ),

    "GCN": GCN(
        data.num_features,
        64,
        dataset.num_classes
    ),

    "GraphSAGE": GraphSAGE(
        data.num_features,
        64,
        dataset.num_classes
    ),

    "GAT": GAT(
        data.num_features,
        16,
        dataset.num_classes,
        heads=4
    ),

    "GIN": GIN(
        data.num_features,
        64,
        dataset.num_classes
    )
}

In [ ]:
results = []

for name, model in models.items():

    print(f"\nTraining {name}...")

    result = train_model(
        model,
        data,
        epochs=200
    )

    result["model"] = name

    results.append(result)

    print(result)

In [ ]:
results_df = pd.DataFrame(results)

results_df = results_df[
    [
        "model",
        "best_val_accuracy",
        "test_accuracy_at_best_val",
        "training_time_seconds",
        "parameters"
    ]
]

results_df = results_df.sort_values(
    "test_accuracy_at_best_val",
    ascending=False
)

results_df

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    results_df["model"],
    results_df["test_accuracy_at_best_val"]
)

plt.ylabel("Test Accuracy")
plt.xlabel("GNN Architecture")
plt.title("Cora Node Classification - GNN Comparison")

plt.xticks(rotation=20)
plt.tight_layout()

plt.show()

In [ ]:
plt.figure(figsize=(10, 5))

plt.bar(
    results_df["model"],
    results_df["training_time_seconds"]
)

plt.ylabel("Training Time (seconds)")
plt.xlabel("GNN Architecture")
plt.title("Cora Training Time Comparison")

plt.xticks(rotation=20)
plt.tight_layout()

plt.show()